# 06 — Tutorial 1: Water Molecular Dynamics Simulation in LAMMPS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module%202%20-%20Molecular%20dynamics/notebooks/06_tutorial_water_lammps.ipynb)

## 🎯 Learning Objectives

- Understand the SPC/E rigid water model and its parameters
- Learn the structure of LAMMPS input scripts and data files
- Build a water simulation box programmatically using Python/NumPy
- Write complete, annotated LAMMPS input files for energy minimization, NVT and NPT runs
- Parse and analyze LAMMPS thermodynamic output with Python
- Compute and interpret the O–O radial distribution function g(r)
- Calculate the self-diffusion coefficient of water from MSD

## 1. The SPC/E Water Model

The **SPC/E** (Extended Simple Point Charge) model (Berendsen *et al.* 1987) is one of the most widely used rigid water models for classical MD simulations.

### 1.1 Geometry

| Parameter | Value |
|-----------|-------|
| O–H bond length | 1.0 Å |
| H–O–H angle | 109.47° |

### 1.2 Non-bonded Parameters

Partial charges (self-polarization correction applied):
$$q_O = -0.8476\,e, \quad q_H = +0.4238\,e$$

Lennard-Jones parameters (oxygen only; hydrogen has no LJ):
$$\varepsilon_{OO} = 0.6502\,\text{kJ/mol}, \quad \sigma_{OO} = 3.166\,\text{Å}$$

### 1.3 Interaction Energy

$$U = \sum_{i<j} \left[ 4\varepsilon_{OO}\left(\left(\frac{\sigma_{OO}}{r_{OO}}\right)^{12} - \left(\frac{\sigma_{OO}}{r_{OO}}\right)^{6}\right) + \frac{1}{4\pi\varepsilon_0}\sum_{\alpha,\beta}\frac{q_\alpha q_\beta}{r_{\alpha\beta}} \right]$$

The molecule is kept **rigid** using the SHAKE/SETTLE algorithm — no bond-stretching or angle-bending terms are needed.

### 1.4 Performance of SPC/E

| Property | SPC/E | Experiment |
|----------|-------|------------|
| Density (g/cm³) | 0.997 | 0.997 |
| D (10⁻⁹ m²/s) | 2.4 | 2.3 |
| ΔH_vap (kJ/mol) | 44.0 | 44.0 |
| T_melt (K) | ~215 | 273 |

> **Note:** SPC/E underestimates the melting point but gives excellent liquid-state thermodynamics near 300 K.

In [ ]:
# =============================================================================
# Ch121a: Molecular Dynamics — Notebook 06: Water MD Tutorial (LAMMPS)
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import cdist
import os

matplotlib_params = {'figure.dpi': 120, 'font.size': 11,
                     'axes.labelsize': 12, 'legend.fontsize': 10}
plt.rcParams.update(matplotlib_params)

print('Packages loaded successfully.')

## 2. Building a Water Box

We generate a simple-cubic lattice of water molecules. Each molecule is oriented with the oxygen at the lattice point and the two hydrogens placed according to the SPC/E geometry.

In [ ]:
def build_water_box(n_side=6, density_g_cm3=0.997):
    """
    Build a cubic water box with n_side^3 molecules on a simple-cubic lattice.
    Returns positions (Angstrom), atom types, charges, and box length.
    """
    # SPC/E geometry
    rOH = 1.0          # Angstrom
    angle_HOH = np.deg2rad(109.47)
    
    # Water molecular mass
    mw_water = 18.015  # g/mol
    N_A = 6.022e23
    
    n_mol = n_side**3
    
    # Box length from density
    mass_g = n_mol * mw_water / N_A
    vol_cm3 = mass_g / density_g_cm3
    L_cm = vol_cm3**(1/3)
    L_ang = L_cm * 1e8  # convert cm to Angstrom
    
    spacing = L_ang / n_side
    
    positions = []  # (atom_id, mol_id, type, x, y, z)
    atom_id = 0
    mol_id = 0
    
    # H positions relative to O, in the xz plane
    half_ang = angle_HOH / 2
    H1_rel = np.array([ rOH * np.sin(half_ang), 0.0,  rOH * np.cos(half_ang)])
    H2_rel = np.array([-rOH * np.sin(half_ang), 0.0,  rOH * np.cos(half_ang)])
    
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                mol_id += 1
                O_pos = np.array([ix, iy, iz]) * spacing + 0.5 * spacing
                
                # Add small random perturbation to avoid perfect lattice artifacts
                np.random.seed(mol_id)
                perturb = (np.random.rand(3) - 0.5) * 0.1
                O_pos += perturb
                
                H1_pos = O_pos + H1_rel
                H2_pos = O_pos + H2_rel
                
                atom_id += 1
                positions.append((atom_id, mol_id, 1, *O_pos))   # O: type 1
                atom_id += 1
                positions.append((atom_id, mol_id, 2, *H1_pos))  # H: type 2
                atom_id += 1
                positions.append((atom_id, mol_id, 2, *H2_pos))  # H: type 2
    
    return positions, n_mol, L_ang

positions, n_mol, L_ang = build_water_box(n_side=6)
n_atoms = len(positions)
print(f'Water box: {n_mol} molecules, {n_atoms} atoms')
print(f'Box length: {L_ang:.4f} Å  ({L_ang/10:.4f} nm)')
print(f'Expected density: ~0.997 g/cm³')

## 3. Writing the LAMMPS Data File

LAMMPS reads molecular topology from a **data file** containing masses, atom coordinates, bonds, and angles.

In [ ]:
def write_lammps_data(filename, positions, n_mol, L_ang):
    """Write a LAMMPS data file for the SPC/E water box."""
    n_atoms = len(positions)
    n_bonds = n_mol * 2   # 2 O-H bonds per molecule
    n_angles = n_mol * 1  # 1 H-O-H angle per molecule
    
    # SPC/E charges
    q_O = -0.8476
    q_H =  0.4238
    
    lines = []
    lines.append('LAMMPS data file — SPC/E water box\n')
    lines.append(f'{n_atoms} atoms')
    lines.append(f'{n_bonds} bonds')
    lines.append(f'{n_angles} angles')
    lines.append('0 dihedrals')
    lines.append('0 impropers\n')
    lines.append('2 atom types')
    lines.append('1 bond types')
    lines.append('1 angle types\n')
    lines.append(f'0.0 {L_ang:.6f} xlo xhi')
    lines.append(f'0.0 {L_ang:.6f} ylo yhi')
    lines.append(f'0.0 {L_ang:.6f} zlo zhi\n')
    lines.append('Masses\n')
    lines.append('1 15.9994  # O')
    lines.append('2  1.0080  # H\n')
    lines.append('Atoms  # full\n')
    for p in positions:
        aid, mol, atype, x, y, z = p
        q = q_O if atype == 1 else q_H
        lines.append(f'{aid} {mol} {atype} {q:.4f} {x:.6f} {y:.6f} {z:.6f}')
    lines.append('')
    lines.append('Bonds\n')
    bond_id = 0
    for m in range(n_mol):
        base = m * 3  # 0-indexed
        O_id  = base + 1
        H1_id = base + 2
        H2_id = base + 3
        bond_id += 1; lines.append(f'{bond_id} 1 {O_id} {H1_id}')
        bond_id += 1; lines.append(f'{bond_id} 1 {O_id} {H2_id}')
    lines.append('')
    lines.append('Angles\n')
    for m in range(n_mol):
        base = m * 3
        O_id  = base + 1
        H1_id = base + 2
        H2_id = base + 3
        lines.append(f'{m+1} 1 {H1_id} {O_id} {H2_id}')
    
    content = '\n'.join(lines)
    with open(filename, 'w') as f:
        f.write(content)
    print(f'Wrote {filename} ({len(lines)} lines)')
    return content

# Write to /tmp for demonstration
data_content = write_lammps_data('/tmp/water.data', positions, n_mol, L_ang)
# Preview first 30 lines
for line in data_content.split('\n')[:30]:
    print(line)

## 4. LAMMPS Input Scripts

A LAMMPS simulation is controlled by an **input script**. We need three scripts:
1. Energy minimization
2. NVT equilibration  
3. NPT production run

### Key LAMMPS Commands

| Command | Purpose |
|---------|--------|
| `units real` | Energy=kcal/mol, length=Å, time=fs |
| `atom_style full` | Supports charges and molecule IDs |
| `pair_style lj/cut/coul/long` | LJ + long-range Coulomb |
| `kspace_style pppm` | Particle-Particle Particle-Mesh Ewald |
| `fix shake` | Rigid bonds/angles (for SPC/E) |
| `fix nvt` | Nosé-Hoover thermostat |
| `fix npt` | Nosé-Hoover thermostat + barostat |
| `thermo` | Print thermodynamic info every N steps |
| `dump` | Write trajectory to file |

In [ ]:
lammps_minimize = """
# ============================================================
# LAMMPS Input: SPC/E Water — Energy Minimization
# ============================================================
units           real
atom_style      full
boundary        p p p

read_data       water.data

# --- Force field ---
pair_style      lj/cut/coul/long 9.0 9.0
pair_coeff      1 1  0.15535 3.166   # O-O: eps(kcal/mol), sigma(Ang)
pair_coeff      1 2  0.0     0.0     # O-H: no LJ
pair_coeff      2 2  0.0     0.0     # H-H: no LJ

bond_style      harmonic
bond_coeff      1  1000.0 1.0        # stiff spring, r0=1.0 Ang

angle_style     harmonic
angle_coeff     1  1000.0 109.47     # stiff spring, theta0=109.47 deg

kspace_style    pppm 1.0e-4

# --- Constraints (rigid SPC/E bonds and angles) ---
fix             shake_water all shake 1.0e-4 20 0 b 1 a 1

# --- Neighbor list ---
neighbor        2.0 bin
neigh_modify    every 1 delay 0 check yes

# --- Minimization ---
minimize        1.0e-4 1.0e-6 1000 10000

write_data      water_min.data
"""

with open('/tmp/in.minimize', 'w') as f:
    f.write(lammps_minimize)
print('Energy minimization input script:')
print(lammps_minimize)

In [ ]:
lammps_nvt = """
# ============================================================
# LAMMPS Input: SPC/E Water — NVT Equilibration (300 K, 100 ps)
# ============================================================
units           real
atom_style      full
boundary        p p p

read_data       water_min.data

pair_style      lj/cut/coul/long 9.0 9.0
pair_coeff      1 1  0.15535 3.166
pair_coeff      1 2  0.0 0.0
pair_coeff      2 2  0.0 0.0

bond_style      harmonic
bond_coeff      1  1000.0 1.0
angle_style     harmonic
angle_coeff     1  1000.0 109.47

kspace_style    pppm 1.0e-4

fix             shake_water all shake 1.0e-4 20 0 b 1 a 1

neighbor        2.0 bin
neigh_modify    every 1 delay 0 check yes

# --- Velocity initialization ---
velocity        all create 300.0 12345 dist gaussian

# --- Nosé-Hoover thermostat: NVT ---
fix             nvt_water all nvt temp 300.0 300.0 100.0

# --- Timestep and run ---
timestep        2.0         # 2 fs (safe with SHAKE)

thermo          100
thermo_style    custom step temp press density etotal ke pe

dump            1 all atom 500 nvt_traj.lammpstrj

run             50000       # 100 ps

write_data      water_nvt.data
write_restart   water_nvt.restart
"""

with open('/tmp/in.nvt', 'w') as f:
    f.write(lammps_nvt)
print('NVT equilibration input script:')
print(lammps_nvt)

In [ ]:
lammps_npt = """
# ============================================================
# LAMMPS Input: SPC/E Water — NPT Production (300 K, 1 atm, 1 ns)
# ============================================================
units           real
atom_style      full
boundary        p p p

read_restart    water_nvt.restart

pair_style      lj/cut/coul/long 9.0 9.0
pair_coeff      1 1  0.15535 3.166
pair_coeff      1 2  0.0 0.0
pair_coeff      2 2  0.0 0.0

bond_style      harmonic
bond_coeff      1  1000.0 1.0
angle_style     harmonic
angle_coeff     1  1000.0 109.47

kspace_style    pppm 1.0e-4
fix             shake_water all shake 1.0e-4 20 0 b 1 a 1

neighbor        2.0 bin
neigh_modify    every 1 delay 0 check yes

# --- Nosé-Hoover NPT: T=300 K, P=1 atm ---
fix             npt_water all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0

timestep        2.0

thermo          1000
thermo_style    custom step temp press density etotal ke pe vol

# Trajectory for analysis (every 100 steps = 200 fs)
dump            1 all custom 100 npt_traj.lammpstrj id mol type x y z vx vy vz
dump_modify     1 sort id

run             500000      # 1 ns

write_data      water_npt_final.data
"""

with open('/tmp/in.npt', 'w') as f:
    f.write(lammps_npt)
print('NPT production input script:')
print(lammps_npt)

## 5. Running LAMMPS

```bash
# After installing LAMMPS (https://lammps.sandia.gov):
lmp -in in.minimize
lmp -in in.nvt
lmp -in in.npt

# Or with MPI parallelization:
mpirun -np 4 lmp -in in.npt
```

> **Note:** The analysis cells below use **synthetic data** that mimics realistic LAMMPS output, so they run without LAMMPS installed. Replace the synthetic data with actual LAMMPS output files for real analysis.

## 6. Parsing LAMMPS Thermodynamic Output

LAMMPS `thermo` output looks like:
```
Step Temp Press Density TotEng KinEng PotEng Volume
0    300.12 -52.3 0.9901 -24100.5 1803.2 -25903.7 18540.2
...
```

In [ ]:
# ---- Generate synthetic LAMMPS thermo output (mimics real output) ----
np.random.seed(42)
n_frames = 500
steps = np.arange(n_frames) * 1000

# Temperature: equilibrates from 320 K to 300 K with fluctuations
tau_equil = 50
T_eq = 300.0 + 20.0 * np.exp(-np.arange(n_frames) / tau_equil)
T_fluct = np.random.normal(0, 2.5, n_frames)  # thermal fluctuations
temperature = T_eq + T_fluct

# Pressure: equilibrates to ~1 atm with large fluctuations
P_eq = 1.0 + 50.0 * np.exp(-np.arange(n_frames) / tau_equil)
P_fluct = np.random.normal(0, 150, n_frames)
pressure = P_eq + P_fluct

# Density: approaches 0.997 g/cm³
rho_eq = 0.997 - 0.02 * np.exp(-np.arange(n_frames) / (2*tau_equil))
rho_fluct = np.random.normal(0, 0.002, n_frames)
density = rho_eq + rho_fluct

# Total energy (kcal/mol per molecule)
E_eq = -9.9 - 0.3 * np.exp(-np.arange(n_frames) / tau_equil)
E_fluct = np.random.normal(0, 0.05, n_frames)
energy_per_mol = E_eq + E_fluct

thermo_data = np.column_stack([steps, temperature, pressure, density, energy_per_mol])
print(f'Synthetic thermo data: {n_frames} frames')
print(f'T_avg (last 400 frames): {temperature[100:].mean():.2f} ± {temperature[100:].std():.2f} K')
print(f'ρ_avg (last 400 frames): {density[100:].mean():.4f} ± {density[100:].std():.4f} g/cm³')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('LAMMPS Thermodynamic Output — SPC/E Water NPT at 300 K, 1 atm', fontsize=13, y=1.01)

# Temperature
ax = axes[0, 0]
ax.plot(steps/1000, temperature, lw=0.7, color='tab:red', alpha=0.7)
ax.axhline(300, ls='--', color='k', lw=1, label='Target 300 K')
ax.axvline(100, ls=':', color='gray', label='Equil. cutoff')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Temperature (K)')
ax.set_title('Temperature')
ax.legend(fontsize=9)

# Pressure
ax = axes[0, 1]
ax.plot(steps/1000, pressure, lw=0.7, color='tab:blue', alpha=0.7)
ax.axhline(1, ls='--', color='k', lw=1, label='Target 1 atm')
ax.axvline(100, ls=':', color='gray')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Pressure (atm)')
ax.set_title('Pressure')
ax.legend(fontsize=9)

# Density
ax = axes[1, 0]
ax.plot(steps/1000, density, lw=0.7, color='tab:green', alpha=0.7)
ax.axhline(0.997, ls='--', color='k', lw=1, label='Expt. 0.997 g/cm³')
ax.axvline(100, ls=':', color='gray')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Density (g/cm³)')
ax.set_title('Density')
ax.legend(fontsize=9)

# Energy
ax = axes[1, 1]
ax.plot(steps/1000, energy_per_mol, lw=0.7, color='tab:purple', alpha=0.7)
ax.axhline(-9.9, ls='--', color='k', lw=1, label='Avg (equil.)')
ax.axvline(100, ls=':', color='gray')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Energy (kcal/mol/molecule)')
ax.set_title('Total Energy per Molecule')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Equilibration visible in first ~100 ps; production from 100-500 ps')

## 7. Radial Distribution Function g(r)

The **radial distribution function** (RDF) describes local structure:

$$g(r) = \frac{V}{N^2} \left\langle \sum_{i \ne j} \delta(r - r_{ij}) \right\rangle \frac{1}{4\pi r^2 \Delta r}$$

For liquid water the **O–O g(r)** has characteristic peaks at:
- ~2.75 Å (first shell, ~4.5 neighbors)
- ~4.5 Å (second shell)
- ~6.8 Å (third shell)

In [ ]:
def compute_rdf(positions_O, L, n_bins=200, r_max=None):
    """
    Compute O-O radial distribution function with minimum image convention.
    positions_O: (N, 3) array of oxygen positions in Angstrom.
    """
    if r_max is None:
        r_max = L / 2.0
    
    N = len(positions_O)
    V = L**3
    rho = N / V
    
    dr = r_max / n_bins
    r_edges = np.linspace(0, r_max, n_bins + 1)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
    counts = np.zeros(n_bins)
    
    for i in range(N):
        for j in range(i+1, N):
            dxyz = positions_O[i] - positions_O[j]
            # Minimum image convention
            dxyz -= L * np.round(dxyz / L)
            r = np.linalg.norm(dxyz)
            if r < r_max:
                idx = int(r / dr)
                if idx < n_bins:
                    counts[idx] += 2  # count both i->j and j->i
    
    # Normalize
    shell_vol = 4/3 * np.pi * (r_edges[1:]**3 - r_edges[:-1]**3)
    g_r = counts / (N * rho * shell_vol)
    
    return r_centers, g_r

# Generate synthetic oxygen positions representing liquid water structure
# (realistic liquid water: ~216 molecules in ~18.6 Å box)
np.random.seed(99)
n_ox = 216
L_water = 18.64  # Å for 216 SPC/E water molecules at 0.997 g/cm³

# Start with random positions
pos_O = np.random.uniform(0, L_water, (n_ox, 3))

# Build synthetic g(r) directly — analytically model the known SPC/E O-O g(r)
r = np.linspace(0.01, 9.3, 500)

# Analytical model: sum of Gaussians representing known peaks
def synthetic_water_gr(r):
    g = np.ones_like(r)
    # Hard core exclusion
    g[r < 2.3] = 0.0
    # First peak at ~2.75 Å
    g += 2.8 * np.exp(-((r - 2.75)/0.12)**2)
    # First minimum at ~3.4 Å
    g -= 0.6 * np.exp(-((r - 3.45)/0.20)**2)
    # Second peak at ~4.5 Å
    g += 0.65 * np.exp(-((r - 4.50)/0.30)**2)
    # Second minimum
    g -= 0.15 * np.exp(-((r - 5.6)/0.35)**2)
    # Third peak at ~6.8 Å
    g += 0.12 * np.exp(-((r - 6.80)/0.45)**2)
    # Ensure non-negative
    g = np.clip(g, 0, None)
    return g

g_OO = synthetic_water_gr(r)
# Add small noise
np.random.seed(7)
g_OO += np.random.normal(0, 0.02, len(r))
g_OO = np.clip(g_OO, 0, None)

print('Synthetic O-O g(r) computed')
print(f'First peak position: {r[np.argmax(g_OO[r>2.4])] + 2.4:.2f} Å (expected ~2.75 Å)')

In [ ]:
# Coordination number: integral of 4πr²ρg(r) dr from 0 to r_min
rho_bulk = n_ox / L_water**3   # Å⁻³
r_min_first_shell = 3.4        # first minimum

mask = r <= r_min_first_shell
dr_val = r[1] - r[0]
n_coord = np.trapz(4 * np.pi * r[mask]**2 * rho_bulk * g_OO[mask], r[mask])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- g(r) ---
ax = axes[0]
ax.plot(r, g_OO, lw=2, color='royalblue', label='SPC/E O-O g(r) (synthetic)')
ax.axhline(1, ls='--', color='gray', lw=1, label='Bulk limit')
ax.axvline(2.75, ls=':', color='red', lw=1.5, label='1st peak ~2.75 Å')
ax.axvline(r_min_first_shell, ls=':', color='orange', lw=1.5, label=f'1st min ~{r_min_first_shell} Å')
ax.axvline(4.50, ls=':', color='green', lw=1.5, label='2nd peak ~4.50 Å')
ax.fill_between(r[mask], g_OO[mask], alpha=0.15, color='royalblue')
ax.set_xlabel('r (Å)')
ax.set_ylabel('g$_{OO}$(r)')
ax.set_title('O–O Radial Distribution Function of SPC/E Water')
ax.legend(fontsize=9)
ax.set_xlim(1.5, 9.3)
ax.set_ylim(-0.05, 3.5)
ax.text(2.75, 3.2, f'$n_{{coord}}$ ≈ {n_coord:.1f}', ha='center', fontsize=11, color='royalblue')

# --- Running coordination number ---
n_run = np.array([
    np.trapz(4*np.pi*r[:i]**2 * rho_bulk * g_OO[:i], r[:i]) if i > 1 else 0
    for i in range(len(r))
])
ax2 = axes[1]
ax2.plot(r, n_run, lw=2, color='darkorange')
ax2.axvline(r_min_first_shell, ls='--', color='gray', lw=1)
ax2.axhline(n_coord, ls=':', color='royalblue', lw=1.5, label=f'n = {n_coord:.1f} at {r_min_first_shell} Å')
ax2.set_xlabel('r (Å)')
ax2.set_ylabel('Running coordination number n(r)')
ax2.set_title('Coordination Number of SPC/E Water')
ax2.legend()
ax2.set_xlim(1.5, 9.3)

plt.tight_layout()
plt.show()
print(f'\nFirst-shell coordination number: {n_coord:.2f} (experiment: ~4.5)')

## 8. Self-Diffusion Coefficient from MSD

The **Einstein relation** gives the diffusion coefficient from the mean-square displacement:

$$D = \lim_{t\to\infty} \frac{\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \rangle}{6t}$$

For SPC/E water at 300 K, $D \approx 2.4 \times 10^{-9}$ m²/s (experiment: $2.3 \times 10^{-9}$ m²/s).

In [ ]:
def generate_water_trajectory(n_mol=50, n_steps=5000, dt_ps=0.2, D_target=2.4e-5):
    """
    Generate synthetic oxygen trajectories for SPC/E-like water.
    D_target in cm²/s (2.4e-5 cm²/s = 2.4e-9 m²/s)
    Returns positions array (n_steps, n_mol, 3) in Angstrom.
    """
    # Convert D from cm²/s to Å²/ps
    # 1 cm²/s = 1e16 Å²/s = 1e4 Å²/ps
    D_ang2_ps = D_target * 1e4  # Å²/ps
    
    # Step size from D = <Δx²> / (2 * dt) per dimension
    sigma_step = np.sqrt(2 * D_ang2_ps * dt_ps)
    
    traj = np.zeros((n_steps, n_mol, 3))
    traj[0] = np.random.uniform(0, 50, (n_mol, 3))  # initial positions
    
    for t in range(1, n_steps):
        traj[t] = traj[t-1] + np.random.normal(0, sigma_step, (n_mol, 3))
    
    return traj

np.random.seed(1234)
traj = generate_water_trajectory(n_mol=100, n_steps=3000, dt_ps=0.2, D_target=2.4e-5)

# Compute MSD
n_steps, n_mol, _ = traj.shape
max_lag = n_steps // 2

msd = np.zeros(max_lag)
for lag in range(1, max_lag):
    disp = traj[lag:] - traj[:-lag]  # (n_frames-lag, n_mol, 3)
    msd[lag] = np.mean(np.sum(disp**2, axis=2))

time_ps = np.arange(max_lag) * 0.2  # ps

# Fit D from linear region (10–50% of total time)
fit_start = int(0.1 * max_lag)
fit_end   = int(0.5 * max_lag)
coeffs = np.polyfit(time_ps[fit_start:fit_end], msd[fit_start:fit_end], 1)
D_fit = coeffs[0] / 6  # Å²/ps
D_fit_m2s = D_fit * 1e-4  # m²/s  (1 Å²/ps = 1e-4 m²/s ... actually need: 1 Å²/ps = 1e-20/1e-12 = 1e-8 m²/s)
# Correct conversion: 1 Å²/ps = (1e-10 m)²/(1e-12 s) = 1e-20/1e-12 = 1e-8 m²/s
D_fit_m2s = D_fit * 1e-8

print(f'Fitted D = {D_fit:.4f} Å²/ps = {D_fit_m2s:.2e} m²/s')
print(f'SPC/E literature: 2.4e-9 m²/s,  Experiment: 2.3e-9 m²/s')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(time_ps[1:], msd[1:], lw=1.5, color='royalblue', label='MSD (synthetic SPC/E oxygen)')
# Fit line
t_fit = time_ps[fit_start:fit_end]
msd_fit = coeffs[0] * t_fit + coeffs[1]
ax.plot(t_fit, msd_fit, '--', color='red', lw=2,
        label=f'Linear fit: slope = {coeffs[0]:.3f} Å²/ps\nD = {D_fit_m2s:.2e} m²/s')
ax.fill_between([time_ps[fit_start], time_ps[fit_end]], [0, 0],
                [msd[fit_end]*1.1, msd[fit_end]*1.1], alpha=0.08, color='red', label='Fit region')
ax.set_xlabel('Time lag (ps)')
ax.set_ylabel('MSD (Å²)')
ax.set_title('Mean-Square Displacement — SPC/E Water Oxygen')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Long-Range Electrostatics: PPPM/Ewald

Coulomb interactions are long-ranged ($1/r$ decay). Naïve cutoff introduces large errors. The **Ewald summation** splits the interaction into:

$$U_{Coulomb} = U_{\text{real-space}} + U_{\text{reciprocal-space}} + U_{\text{self}}$$

- **Real space:** Short-ranged, screened by $\text{erfc}(\alpha r)/r$
- **Reciprocal space (k-space):** Long-range part via Fourier transform

$$U_{\text{recip}} = \frac{1}{2V\varepsilon_0} \sum_{\mathbf{k}\ne 0} \frac{e^{-k^2/4\alpha^2}}{k^2} |S(\mathbf{k})|^2$$

**PPPM** (Particle-Particle Particle-Mesh) interpolates charges onto a mesh and uses FFT → $\mathcal{O}(N \log N)$ scaling.

In LAMMPS: `kspace_style pppm 1.0e-4` (accuracy = $10^{-4}$ kcal/mol).

In [ ]:
# Illustrate: Ewald real-space screening function vs. plain Coulomb
r_plot = np.linspace(0.01, 12, 500)
alpha = 0.3      # Ewald splitting parameter (Å⁻¹)

from scipy.special import erfc

V_coulomb  = 1.0 / r_plot                         # plain 1/r
V_real     = erfc(alpha * r_plot) / r_plot         # real-space screened
V_recip    = (1 - erfc(alpha * r_plot)) / r_plot   # reciprocal-space part (handled by k-space)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_plot, V_coulomb, 'k-',  lw=1.5, label='Full Coulomb 1/r')
ax.plot(r_plot, V_real,    'b-',  lw=2,   label='Real-space (erfc/r), cutoff at ~10 Å')
ax.plot(r_plot, V_recip,   'r--', lw=2,   label='Reciprocal-space part (→ PPPM mesh)')
ax.axvline(1/alpha * 3, ls=':', color='gray', label=f'Cutoff ≈ {1/alpha*3:.1f} Å')
ax.set_xlim(0, 12); ax.set_ylim(-0.1, 1.5)
ax.set_xlabel('r (Å)')
ax.set_ylabel('V(r) (arb. units)')
ax.set_title(f'Ewald Splitting (α = {alpha} Å⁻¹)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Summary and Workflow

### Complete LAMMPS Water Simulation Workflow

```
1. Build water box (Python/NumPy or Packmol)
       ↓
2. Write LAMMPS data file  (water.data)
       ↓
3. Energy minimization     (lmp -in in.minimize)
       ↓
4. NVT equilibration       (lmp -in in.nvt,  ~100 ps)
       ↓
5. NPT production run      (lmp -in in.npt,  ~1 ns)
       ↓
6. Analysis:
   - Parse thermo output   → T, P, ρ convergence
   - Compute O-O g(r)      → structure, coordination number
   - Compute O MSD         → self-diffusion D
   - Compute VACF          → vibrational density of states
```

### Key Results for SPC/E Water at 300 K, 1 atm

| Property | SPC/E | Experiment |
|----------|-------|------------|
| Density (g/cm³) | 0.997 | 0.997 |
| 1st O-O peak (Å) | 2.75 | 2.80 |
| Coordination number | ~4.5 | ~4.5 |
| D (10⁻⁹ m²/s) | 2.4 | 2.3 |
| ΔH_vap (kJ/mol) | 44.0 | 44.0 |

### References

1. Berendsen, H. J. C. *et al.* J. Phys. Chem. **1987**, 91, 6269 (SPC/E model)
2. LAMMPS Documentation: https://docs.lammps.org
3. Vega, C. & Abascal, J. L. F. Phys. Chem. Chem. Phys. **2011**, 13, 19663 (water models review)